In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import pandas as pd
import json
from database.querys import get_movies
from database.connection import get_engine
from services.tmdb_services import get_movie_details, get_movie_credits
from services.movies_service import string_to_list
from database.querys import get_categories
from sqlalchemy import text

# 🔍Un primer vistazo a los datos

### Obtenemos los datos de la base de datos

In [ ]:
df_movies = get_movies()
df_movies.head()


### Cantidas de Filas y columnas

In [ ]:
display(df_movies.shape)

### Columnas de la tabla movies

In [ ]:
display(df_movies.columns)

### Informacion de cada columna

In [ ]:
display(df_movies.info())

### Valores `null` por columnas

In [ ]:
display(df_movies.isnull().sum())

### Año de lanzamiento de la pelicula mas antigua
Este dato es utilizado en el filtro de rango de años

In [ ]:
years = df_movies['release_date'].dropna().apply(lambda x: x.year)
older_year = int(years.min())
display(f"Año de lanzamiento de la pelicula más antigua: {older_year} ")

### Top 3 peliculas mas populares

In [ ]:
display(pd.DataFrame.head(df_movies.sort_values(by='popularity', ascending=False), 3))

### Generos de las peliculas

In [ ]:
categorias = get_categories()
display(categorias)

### Categoria mas repetidas

In [ ]:
genre_lists = df_movies['genre_ids'].dropna().apply(string_to_list)

genre_counts = (
    pd.Series([gid for sublist in genre_lists for gid in sublist])
    .value_counts()
    .rename_axis('category_id')
    .reset_index(name='count')
)

top_category_id = int(genre_counts.loc[0, 'category_id'])
categoria = categorias[(categorias['id'] == top_category_id)]
display(categoria)

### Buscar categoria por `id` en la base de datos

In [ ]:
engine = get_engine()

def get_category_name(category_id):
    query = text("""
                SELECT name
                FROM categories
                WHERE id = :category_id
                """)
    params = {"category_id":category_id}
    
    result = pd.read_sql(query, engine, params=params)
    
    if not result.empty:
        return result.iloc[0]['name']
    
    else:
        return None

get_category_name(53)

### Funcion para convertir string con del tipo {1,2,3} a `list`
Columnas como `genre_ids` son de este tipo

In [ ]:
def string_to_list(cadena):
    return [int(x) for x in cadena.strip('{}').split(',') if x]

df_0 = df_movies['genre_ids'][0]
list_0 = string_to_list(df_0)
display(list_0)

### Idiomas originales disponibles
Listado de codigos en la columna `original_language`.

In [ ]:
languages = df_movies['original_language'].dropna().unique()
languages

### Ver detalles de una pelicula por `id`

In [ ]:
details = get_movie_details(1297842)
details

### Crea un archivo `json` para ver todos los detalles de la pelicula obtenida

In [ ]:
with open("movie.json", "w", encoding="utf-8") as file:
    json.dump(details, file, indent=4, ensure_ascii=False)

### Obtiene los creditos de una pelicula por `id`
En los creditos podemos encontrar datos como el `cast` y `director`

In [ ]:
credits = get_movie_credits(1318447)
credits

### Obtener el director

In [ ]:
crew = credits['crew']
director = next(
    (person["name"] for person in crew if person["job"] == "Director"),
    None
)
director

### Obtener los 5 primeros actores del cast

In [ ]:
cast = credits['cast']
top_cast = [actor['name'] for actor in cast[:5]]
top_cast